# 7장. Function Tool로 업무 데이터 연결하기

기본 CSV 보고서에 애플리케이션의 월간 목표 조회를 연결합니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | Function Tool 등록과 공식 SDK의 호출 처리 |
| 이번 실습 조합 | CSV 분석 + 로컬 업무 함수 + 목표·달성률 응답 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

06장에서는 sandbox 패키지를 준비했습니다. 이 장은 sandbox의 CSV 분석에 **현재 Python 애플리케이션에서 실행하는 함수**를 연결합니다. 업무 데이터는 합성 JSON이며 실제 사내 서비스나 자격 증명을 사용하지 않습니다.

에이전트가 목표 조회 요청 → 공식 SDK가 인자 검증 → 애플리케이션 함수 실행 → SDK가 결과 제출 → 에이전트가 목표를 활용해 분석·보고서 작성 순서로 진행합니다.

| 작업 | 실행 위치 |
| --- | --- |
| CSV 분석·보고서 작성 | OpenAI-hosted sandbox |
| `get_sales_target()` | 노트북의 로컬 또는 Colab Python |
| 인자 검증·함수 호출·결과 제출 | 애플리케이션의 공식 SDK |
| 최종 응답·파일 읽기 | 노트북 |

sandbox 네트워크 설정은 로컬 함수의 실행 위치나 네트워크 권한을 바꾸지 않습니다. 기본 실습은 SDK 자동 처리를 사용하며 직접 pending action 응답은 심화 자료로 분리합니다.

수동 실습은 위에서 아래로 실행합니다. 자동 검증은 `uv run --group dev python scripts/run_extensions.py --chapter 7 --live`로 실행합니다.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 환경 준비와 실행 모드

로컬에서는 `uv sync --locked` 후 `.venv` 커널을 선택합니다. Colab에서는 다운로드 안내의 독립 실행본을 업로드합니다. 준비 셀은 패키지를 직접 설치하고 포함 파일을 생성합니다.
아래 RUN_LIVE를 True로 바꿨을 때만 실제 API 호출·sandbox 생성이 발생합니다.
기존 `.env.local`의 키·모델을 재사용하며 키는 sandbox·ZIP·로그에 넣지 않습니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
import json
from tutorial_helpers.common import load_settings

RUN_LIVE = False
MODEL_OVERRIDE = ""
CHAPTER = "07_function_tools"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print({"live_enabled": settings.enabled, "model": settings.model or "Offline mode"})


## 1. CSV 입력과 업무 함수 준비하기

CSV의 매출은 sandbox에서 Python으로 계산하고, 목표 매출은 애플리케이션 함수가 로컬 `data/sales_targets.json`에서 읽습니다. 목표 파일을 sandbox에 업로드하거나 목표값을 프롬프트에 미리 넣지 않습니다.

아래 로컬 파일 읽기는 API 호출이 아닙니다. CSV는 다음 요청의 inline 파일로 공급합니다.


In [ ]:
input_path = settings.root / "data/sales.csv"
csv_bytes = input_path.read_bytes()
csv_text = csv_bytes.decode("utf-8")
print(csv_text)


### 업무 함수 정의하기

함수의 입력은 모델이 선택하는 `period` 하나이며 반환값은 Python 딕셔너리입니다. 역할은 애플리케이션의 `ACTOR_ROLE`에서 확인하고 모델 인자로 받지 않습니다. 이 고정 역할 값은 학습용이며 실제 인증 시스템은 아닙니다.

일반 Python 함수의 타입 주석만으로 런타임 인자 검증이 추가되지는 않습니다. 뒤에서 만드는 SDK 어댑터가 인자 스키마를 검증하고, 함수 본문은 별도로 허용 역할·기간을 검사합니다.


In [ ]:
from tutorial_helpers.extensions import PolicyError

ACTOR_ROLE = "analyst"


def get_sales_target(period: str) -> dict:
    """Look up a fictional monthly sales revenue target, e.g. period 2026-01."""
    if ACTOR_ROLE != "analyst":
        raise PolicyError("role_not_allowed")
    values = json.loads((settings.root / "data/sales_targets.json").read_text())
    if period not in values:
        raise PolicyError("unknown_period")
    return {"period": period, **values[period]}


### 로컬 사전 확인과 SDK 도구 생성

다음 직접 호출은 함수를 확인하는 **로컬 사전 점검**이며 Agents API가 도구를 호출한 증거가 아닙니다. 이후 API 실행에서 관찰한 도구 호출 수와 구분합니다.

`function_tool()`은 타입 주석·docstring으로 도구 설명과 인자 스키마를 만들고 로컬 핸들러를 묶습니다.

| 사용 방식 | 반환·역할 |
| --- | --- |
| `get_sales_target("2026-01")` | Python 딕셔너리 |
| `sales_target_tool`에 인자 딕셔너리 전달 | SDK 인자 검증·함수 호출·직렬화 |
| `sales_target_tool.definition` | 모델에 전달할 도구 설명·스키마 |

이미 SDK가 직렬화한 도구 출력을 다시 `json.dumps()`하지 않습니다. 정상 API 실행에서 결과 제출은 SDK에 맡깁니다.


In [ ]:
print("Local preflight only:", get_sales_target("2026-01"))


In [ ]:
from openai.lib.beta.agents import function_tool

sales_target_tool = function_tool(get_sales_target)
print("SDK tool definition:")
print(json.dumps(sales_target_tool.definition, indent=2))


## 2. 도구 등록과 분석 요청 구성하기

| 설정 | 전달할 값 | 역할 |
| --- | --- | --- |
| `agent.tools` | `sales_target_tool.definition` | 모델이 호출할 도구 설명·인자 스키마 |
| `tool_handlers` | 이름과 `sales_target_tool`의 매핑 | 모델 요청을 로컬 함수로 연결 |
| `environment.files` | CSV inline 데이터 | sandbox의 분석 입력 |

공식 SDK가 함수 호출 Item을 처리하여 인자 검증·로컬 함수 실행·직렬화·원래 Turn/Call에 결과 제출을 담당합니다. 직접 `tool_result`를 보내는 코드를 같은 호출에 중복 적용하지 않습니다.

[Function Tool 심화 자료](../../docs/extensions/function-tool-actions.md)는 직접 pending 응답과 부작용 있는 함수의 결과 캐시를 설명합니다. 이 장의 읽기 전용 조회와 11장의 승인·내구성 캐시는 서로 다른 범위입니다.


In [ ]:
import base64

agent = {
    "model": settings.model,
    "instructions": (
        "You are a careful sales analyst. Use Python for all calculations. "
        "Use only the provided CSV; never invent rows. Do not browse or delegate. "
        "Keep progress commentary short. Return only the requested JSON, without Markdown fences."
    ),
    "tools": [sales_target_tool.definition],
}
handlers = {sales_target_tool.name: sales_target_tool}
environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "disabled"},
    "files": [{"type": "inline", "path": "/workspace/sales.csv",
               "data": base64.b64encode(csv_bytes).decode("ascii")}],
}
print({"model_tool_names": [tool["name"] for tool in agent["tools"]],
       "local_handler_names": list(handlers),
       "sandbox_input": environment["files"][0]["path"]})


### 최종 응답과 파일의 형식 구분

| 결과 | 포함할 내용 |
| --- | --- |
| 최종 JSON | `metrics`, `target_revenue`, `attainment_rate` |
| `summary.json` | 기본 매출 집계만 |
| `report.md` | 매출·목표·달성률을 한국어로 표시 |

`METRICS_CONTRACT`는 내부 `metrics`의 구조이고 `RESULT_CONTRACT`는 최종 확장 응답의 구조입니다. 기본 객체만 반환하라는 지시와 확장 객체 반환 지시를 충돌시키지 않습니다. 목표값은 실제 도구 결과에서 얻도록 요청합니다.


In [ ]:
METRICS_CONTRACT = (
    'The metrics object must contain exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": integer, '
    '"by_product": {product_name: integer_revenue}}. '
    "Revenue is quantity multiplied by unit_price. Do not round integer amounts. "
    "Each by_product value must be integer revenue, never a nested object."
)
RESULT_CONTRACT = (
    "Return only a JSON object with keys metrics (the base object), "
    "target_revenue (integer), and attainment_rate "
    "(total_revenue / target_revenue as a unit fraction, not multiplied by 100)."
)
task = (
    "Read /workspace/sales.csv and calculate sales metrics using Python. "
    "Call get_sales_target for period 2026-01 exactly once and use the actual tool result. "
    "Write /workspace/outputs/summary.json with only the base metrics. "
    "The final metrics object must equal the parsed summary.json object. "
    "Write /workspace/outputs/report.md in Korean beginning with '# 매출 분석 보고서'; "
    "include all metrics, the target, and attainment percentage. "
    + METRICS_CONTRACT + " " + RESULT_CONTRACT
)


## 3. 도구를 등록해 Agents API 실행하기

앞에서 구성한 `agent`와 `environment`를 전달하고 `tool_handlers=handlers`로 로컬 실행을 연결합니다. 일반 텍스트 응답과 Function Call Item을 표시하며, 실제 완료한 호출 이력은 다음 절에서 조회합니다.

각 실행·결과 단계는 오류·일반적인 인터럽트에서 `cleanup_on_error()`로 정리를 시도합니다. 정상 경로에서는 **7절 정리 셀까지 실행**하고 중간에 학습을 멈출 때도 정리 셀을 실행하세요. 강제 커널 종료는 자동 정리를 보장하지 못합니다.


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

기존 실습 경로 `outputs/extensions/<chapter>/<run_id>/`에 복구 기록을 보관합니다. 다운로드는 그 아래 `downloads/<turn_id>/`에 저장합니다. 상태 준비 셀은 새 실행 시작 시 한 번만 실행하세요. 진행 중 재실행하면 자원 식별자를 잃을 수 있습니다.

타이머는 API 실행 직전에 시작합니다. 600초 제한은 이벤트 도착 시 확인하므로 무응답 중 강제 중단하는 전체 타이머가 아닙니다. 클라이언트 HTTP timeout은 90초입니다.


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import time


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/extensions" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
result = turn = None
downloads = None
downloads_complete = False


### 세션 생성과 자동 도구 처리 관찰

새 작업의 중복 제출은 차단합니다. SDK가 함수 실행과 결과 제출을 처리하며 본문은 별도 pending 응답 루프를 만들지 않습니다. 표시된 Function Call Item 자체를 함수 성공으로 판단하지 말고 다음 절의 저장 상태와 구분하세요.

`with_result_collection()`으로 응답을 모아 `get_final_result()`로 받고, 스트림을 닫은 뒤 root Turn을 조회합니다. 이벤트 수·순서를 고정하거나 `idle`만으로 성공을 판단하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if attempts:
        raise RuntimeError("This run already attempted session creation. Do not resubmit.")
    started = time.monotonic()
    with client.beta.agents.sessions.create(
        agent=agent,
        environment=environment,
        input=task,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
        stream=True,
        tool_handlers=handlers,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.turn.item.added" and event.item.type == "function_call":
                print("Function call item:", event.item.name, "Call ID:", event.item.call_id)
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        result = stream.get_final_result()
        sessions.setdefault(result.session_id, {"deleted": False})
    turn = client.beta.agents.sessions.turns.retrieve(
        result.turn_id, session_id=result.session_id, timeout=15,
    )
    if result.turn.status != "completed" or turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    print("\nCompleted turn:", {"session_id": result.session_id, "turn_id": turn.id,
                                 "status": turn.status})


## 4. 이번 Turn의 실제 함수 호출 이력 확인하기

`items.list()`를 SDK 페이지 반복으로 읽고 이번 root Turn의 `function_call`만 표시합니다. 함수 이름·인자·Call ID·상태와 목표 함수 호출 기록 수를 확인하세요. 로컬 사전 점검 호출은 이 이력에 포함되지 않습니다.

프롬프트의 '한 번 호출'은 요청이지 실행 보장이 아닙니다. 별도 검증기는 정확한 함수명·기간·완료 상태와 호출 1회를 검사합니다. 이것은 장애·프로세스 재시작을 넘어선 exactly-once 보장이나 내구성 캐시 검증이 아닙니다. SDK 스트림 내부 중복 처리와 11장의 내구성 캐시를 구분하세요.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before inspecting tool history.")
    items = list(client.beta.agents.sessions.items.list(
        result.session_id, order="asc", limit=100, timeout=15,
    ))
    function_calls = [item for item in items
                      if item.type == "function_call" and item.turn_id == result.turn_id]
    print("Function calls from this turn:")
    for call in function_calls:
        arguments = json.loads(call.arguments) if isinstance(call.arguments, str) else call.arguments
        print({"name": call.name, "arguments": arguments, "call_id": call.call_id,
               "turn_id": call.turn_id, "status": call.status})
    print({"get_sales_target_calls": sum(c.name == sales_target_tool.name for c in function_calls)})


## 5. 정확한 Turn의 결과 파일 다운로드하기

05장처럼 `artifacts.for_result(result)`로 결과의 Session·Turn에 범위를 고정하고 정확한 경로를 다운로드합니다. SDK가 누락·중복 경로를 오류로 처리합니다. 반환된 artifact와 실제 로컬 경로·크기를 표시합니다.

두 파일을 모두 받아야 완료로 표시합니다. 재실행은 세션 정리 전에 한정하며 부분 파일을 성공으로 처리하지 않습니다. 오류 시 정리를 시도하므로 같은 세션에서 다운로드 재시도를 보장하지 않고 분석을 자동 재실행하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before downloading artifacts.")
    downloads_complete = False
    downloads = output_dir / "downloads" / result.turn_id
    downloads.mkdir(parents=True, exist_ok=True)
    scoped = client.beta.agents.sessions.artifacts.for_result(result)
    for name in ("summary.json", "report.md"):
        local_path = downloads / name
        artifact = scoped.download(f"/workspace/outputs/{name}", to=local_path, timeout=30)
        print({"artifact_id": artifact.id, "turn_id": artifact.turn_id,
               "sandbox_path": artifact.path, "local_path": str(local_path),
               "local_bytes": local_path.stat().st_size})
    downloads_complete = True


## 6. 확장 응답과 내려받은 파일 읽기

최종 응답의 `metrics`는 기본 매출 집계이고 `target_revenue`·`attainment_rate`는 도구 결과를 활용한 확장 정보입니다. `summary.json`에는 기본 집계만 있습니다. 두 구조를 같은 JSON으로 취급하지 않습니다.

JSON의 달성률은 비율이며 예를 들어 `0.85`는 보고서의 `85%`와 같은 표현입니다. 실제 값은 도구 결과와 CSV로 계산하도록 요청했고 미리 프롬프트에 넣지 않았습니다.

아래 정상 경로는 최종 JSON·로컬 JSON을 파싱하고 보고서를 표시합니다. JSON 파싱, 사람의 출력 확인, 별도 검증기의 계산값·보고서 정답 대조는 서로 다른 확인입니다. 내려받은 파일은 세션 삭제 후에도 읽을 수 있습니다.


In [ ]:
from IPython.display import Markdown, display

with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if not downloads_complete:
        raise RuntimeError("Complete both downloads before reading local results.")
    actual = json.loads(result.output_text)
    downloaded_summary = json.loads((downloads / "summary.json").read_text(encoding="utf-8"))
    report_text = (downloads / "report.md").read_text(encoding="utf-8")
    print("Final enriched response:")
    print(json.dumps(actual, indent=2))
    print("Downloaded base summary.json:")
    print(json.dumps(downloaded_summary, indent=2))
    display(Markdown(report_text))


## 7. 세션 정리와 로컬 결과 보관

중간에 학습을 멈추더라도 정리 셀을 실행하세요. 필요한 파일은 삭제 전에 내려받습니다. helper는 이번 실행의 metadata로 소유 세션을 찾아 필요하면 취소한 뒤 삭제·재조회 `404`로 API상 부재를 확인합니다.

출력의 `deleted`·`verified_absent`를 확인하세요. 물리적 sandbox 정리의 동기 완료를 뜻하지 않습니다. 기존 실행·검증 오류가 있으면 정리 오류가 원래 원인을 가리지 않도록 처리합니다.

검증 도구의 실행 사본은 `outputs/extensions-validation/`, 결과·정리 기록은 `outputs/extensions/`에 저장합니다. 수동 Jupyter 실행에는 정답 검사·receipt를 추가하지 않습니다. 로컬 함수 정의·결과·다운로드 파일은 세션 삭제 후에도 남으며 이 장은 별도 저장 Agent를 만들지 않습니다.


In [ ]:
try:
    print("Cleaning up this run's session...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({"session_cleanup": sessions})


## 8. 오류 처리와 복구 범위

| 상황 | 기본 실습의 동작·확인 |
| --- | --- |
| 잘못된 인자 구조 | SDK 어댑터가 함수 실행 전에 검사 |
| 역할·허용 기간 위반 | 함수가 `PolicyError` 발생 |
| 함수 인자·실행·출력 오류 | SDK가 일반 실패 결과를 제출. 원래 예외 본문은 모델에 보내지 않음 |
| 도구 등록 누락 | 도구 이름·스키마와 로컬 handler 매핑 확인 |
| API·스트림 오류 | 함수 오류와 구분하여 정리 결과·저장된 ID 확인 |

현재 SDK 자동 처리의 실패 결과는 `success=False`와 일반 오류 문구입니다. `unknown_period`·`role_not_allowed`가 그대로 모델에 전달된다고 가정하지 않습니다.

**선택 진단:** 현재 고정 SDK의 `on_tool_error`는 후속 입력용 `sessions.stream()`에서 지원하며, 본문의 `sessions.create()`에는 해당 인자가 없습니다. 아래는 후속 Turn을 구성하고 진단이 필요할 때만 참고하는 실행하지 않는 코드입니다. 원래 예외 본문 대신 함수명·단계·예외 종류만 표시합니다. API·스트림 오류용 콜백은 아닙니다.

```python
def show_tool_error(error):
    print({"tool": error.tool_name, "stage": error.stage,
           "error_type": type(error.error).__name__})

# Optional for a later turn: pass on_tool_error=show_tool_error to sessions.stream(...).
```

스트림이 끊겼다고 함수를 무조건 다시 실행하거나 도구 결과를 수동 중복 제출하지 않습니다. 기본 예제는 오류 시 정리를 시도하며 중단 후 자동 복구를 구현하지 않습니다. 강제 종료 시 `outputs/extensions/<chapter>/<run_id>/creation.json`과 Session ID로 이번 실행만 식별하세요. 키나 원본 오류 body는 출력하지 않습니다.

직접 pending 응답·내구성 캐시는 [심화 자료](../../docs/extensions/function-tool-actions.md)와 11장을 참고하세요. [14장 관찰자 재연결](14_operations.ipynb)은 Function Tool이 없는 기본 실행을 다루므로 도구 복구까지 검증했다고 해석하지 않습니다.


## 완료 기준과 다음 장

함수 스키마와 로컬 핸들러 등록, 실제 API 호출 이력, 확장 응답과 기본 집계 파일의 차이, 세션 정리 결과를 설명할 수 있어야 합니다. 자동 검증은 실제 호출 조건과 도구 결과·CSV 기준값을 대조하며, 사전 직접 호출만으로 API 도구 실행을 확인했다고 판단하지 않습니다.

[8장 MCP](08_mcp.ipynb)에서는 같은 월간 목표 조회를 MCP 서버로 제공합니다. 이 장의 애플리케이션 함수 등록과 MCP 도구 연결의 차이를 이어서 배웁니다.


### 기본 스킬이 출력에 등장하는 이유

OpenAI-hosted 이미지에는 사용자 등록과 별개로 기본 스킬이 있을 수 있습니다. CSV는 Spreadsheets의 적용 대상입니다. 모델의 언급만으로 실제 사용을 확정하지 않으며, 기본 목록과 확인 방법은 [기본 스킬 보충 자료](../../docs/extensions/builtin-skills.md)를 참고하세요.


## 공식 참고 자료

### Agents API 전용 문서

- [Functions](https://developers.openai.com/api/docs/guides/agents-api/tools/functions) — 함수 정의·required action·도구 결과 반환과 복구.
- [Events and items](https://developers.openai.com/api/docs/guides/agents-api/sessions/events) — 이벤트 스트림·완료 상태·저장된 items와 turns 조회.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
